<a href="https://colab.research.google.com/github/bhaveshneekhra/synthpop/blob/main/combined_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/UDST/synthpop.git
!cd synthpop
!git checkout aa6c4e836e5b06e78d1a6d897768500f6c16e45b
!cd ..
!pip install -r synthpop/requirements-dev.txt
!python synthpop/setup.py develop

!pip3 install geopandas

!wget https://data.worldpop.org/GIS/Population_Density/Global_2000_2020_1km/2020/IND/ind_pd_2020_1km_ASCII_XYZ.zip
!unzip ind_pd_2020_1km_ASCII_XYZ.zip

Cloning into 'synthpop'...
remote: Enumerating objects: 1788, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 1788 (delta 141), reused 228 (delta 135), pack-reused 1551
Receiving objects: 100% (1788/1788), 1006.67 KiB | 2.09 MiB/s, done.
Resolving deltas: 100% (1105/1105), done.
fatal: not a git repository (or any of the parent directories): .git
     |████████████████████████████████| 42 kB 435 kB/s 
     |████████████████████████████████| 51 kB 365 kB/s 
     |████████████████████████████████| 2.8 MB 8.9 MB/s 
     |████████████████████████████████| 207 kB 47.4 MB/s 
     |████████████████████████████████| 3.1 MB 39.0 MB/s 
     |████████████████████████████████| 90 kB 9.3 MB/s 
     |████████████████████████████████| 121 kB 59.3 MB/s 
     |████████████████████████████████| 100 kB 9.2 MB/s 
     |████████████████████████████████| 84 kB 3.3 MB/s 
  Attempting uninstall: sphinx
    Found existing installation: Sphi

running develop
running egg_info
creating synthpop.egg-info
writing synthpop.egg-info/PKG-INFO
writing dependency_links to synthpop.egg-info/dependency_links.txt
writing requirements to synthpop.egg-info/requires.txt
writing top-level names to synthpop.egg-info/top_level.txt
writing manifest file 'synthpop.egg-info/SOURCES.txt'
reading manifest file 'synthpop.egg-info/SOURCES.txt'
writing manifest file 'synthpop.egg-info/SOURCES.txt'
running build_ext
Creating /usr/local/lib/python3.7/dist-packages/synthpop.egg-link (link to .)
Adding synthpop 0.1.1 to easy-install.pth file

Installed /content
Processing dependencies for synthpop==0.1.1
Searching for us>=0.8
Reading https://pypi.org/simple/us/
Best match: us 2.0.2
Processing us-2.0.2.tar.gz
Writing /tmp/easy_install-pkiyxzut/us-2.0.2/setup.cfg
Running us-2.0.2/setup.py -q bdist_egg --dist-dir /tmp/easy_install-pkiyxzut/us-2.0.2/egg-dist-tmp-8r9zidl1
zip_safe flag not set; analyzing archive contents...
Moving us-2.0.2-py3.7.egg to /usr/

In [ ]:
TINY = 1e-20
POSSIBLE_JOB_LABELS = [
                        'Accountants',
                        'Ag labour',
                        'Air/ship officers',
                        'Artists',
                        'Assemblers',
                        'Barbers',
                        'Boilermen',
                        'Book-keepers',
                        'Carpenters',
                        'Chemical',
                        'Cinema op',
                        'Clerical nec',
                        'Clerical supe',
                        'Computing op',
                        'Construction',
                        'Cooks/waiters',
                        'Cultivators',
                        'Drivers',
                        'Economists',
                        'Elected officials',
                        'Electrical',
                        'Eng. tech',
                        'Engineers',
                        'FIRE sales',
                        'Farm manager',
                        'Fishermen',
                        'Food',
                        'Forestry',
                        'Govt officials',
                        'Homebound',
                        'Hotel/restaurant',
                        'House keepers',
                        'Hunters',
                        'Jewellery',
                        'Journalists',
                        'Labour nec',
                        'Launderers',
                        'Lawyers',
                        'Life science tech',
                        'Life scientists',
                        'Loaders',
                        'Machine tool op',
                        'Maids',
                        'Mail distributors',
                        'Managerial nec',
                        'Manuf agents',
                        'Metal workers',
                        'Mgr Whsl/retail',
                        'Mgr finance',
                        'Mgr manf',
                        'Mgr service',
                        'Mgr transp/commun',
                        'Miners',
                        'Money lenders',
                        'Nursing',
                        'Other farm',
                        'Other farmers',
                        'Other scientific',
                        'Painters',
                        'Paper',
                        'Performers',
                        'Physical sci tech',
                        'Physicians',
                        'Plantation lab',
                        'Plumbers/welders',
                        'Police',
                        'Potters',
                        'Printing',
                        'Production nec',
                        'Professional nec',
                        'Rubber/plastic',
                        'Sales, nec',
                        'Sales, shop',
                        'Service nec',
                        'Shoe makers',
                        'Shopkeepers',
                        'Social scientists',
                        'Statisticians',
                        'Stone cutters',
                        'Sweepers',
                        'Tailors',
                        'Tanners',
                        'Teachers',
                        'Technical sales',
                        'Telephone op',
                        'Textile',
                        'Tobacco',
                        'Transp conductors',
                        'Transp/commun supe',
                        'Typists',
                        'Unidentifiable occ',
                        'Village officials',
                        'Wood/paper'                      
]
                       


In [ ]:
from synthpop.synthpop.zone_synthesizer import synthesize_all_zones, load_data #From https://github.com/UDST/synthpop.git
import pandas as pd
import numpy as np
import random
from io import StringIO
import geopandas as gpd
from shapely.geometry import Point, MultiPoint
from shapely.ops import unary_union
import gc
import os

from tqdm import tqdm
tqdm.pandas()

###Helper functions for data cleaning
def try_convert(value, default, *types):
	for t in types:
		try:
			return t(value)
		except (ValueError, TypeError):
			continue
	return default

def get_probabilistic_place_assignment_batch(individuals_geocode, places_geocode):
    individuals_geocode_sq_sum = np.power(np.linalg.norm(individuals_geocode, axis=1, keepdims=True),2)
    workplaces_geocode_sq_sum = np.power(np.linalg.norm(places_geocode, axis=1, keepdims=True),2)
    inverse_distances = 1/(individuals_geocode_sq_sum+workplaces_geocode_sq_sum.T-2*np.matmul(individuals_geocode, places_geocode.T)+TINY)
    inverse_distances = inverse_distances/inverse_distances.sum(axis=1,keepdims=True)
    return (inverse_distances.cumsum(axis=1)>np.random.uniform(size=(individuals_geocode.shape[0],1))).argmax(axis=-1)

def get_probabilistic_place_assignment(individuals_geocode, places_geocode, batch_size=10000):
	n_batches = int(np.ceil(len(individuals_geocode)/batch_size))
	batch_wise_indicies = []
	for batch_counter in range(n_batches):
		batch_wise_indicies.append(get_probabilistic_place_assignment_batch(individuals_geocode[(batch_counter)*batch_size:(batch_counter+1)*batch_size], places_geocode))
	return np.concatenate(batch_wise_indicies)

class IPU():
	def __init__(self):
		pass

	def preprocess_individual_samples_ihds(self, filtered_ihds_individuals_data):

		# columns_to_keep_individuals = ['DISTRICT', 'IDHH', 'PERSONID', 'RO3', 'RO6', 'RO5','ED2', 'ID11', 'ID13', 'RO7', 'URBAN2011']
		columns_to_keep_individuals = ['DIST01','IDHH','PERSONID', 'RO3', 'RO5', 'ID11', 'ID13']
		columns_rename_dict_individuals = {'RO3':'SexLabel', 'RO5':'Age', 
			'RO6':'marital_status',
			'ED2':'literacy', 'ED6':'edu_years', 'EDUC7': 'edu_cat',
			'ID11':'religion', 'ID13':'caste', 
			'URBAN2011':'residence',
			'WS4':'job', 
			'RO7':'activity_status', 
			'IDHH':'serialno', 
			'PERSONID':'mem_id',
			'DIST01':'district', 
			'MB3':'M_Cataract', 'MB4':'M_TB', 'MB5':'M_High_BP',
			'MB6':'M_Heart_disease', 'MB7':'M_Diabetes', 'MB8':'M_Leprosy',
			'MB9':'M_Cancer', 'MB10':'M_Asthma', 'MB11':'M_Polio',
			'MB12':'M_Paralysis', 'MB13':'M_Epilepsy', 'SM4':'M_Fever', 'SM5':'M_Cough',
			'SM7':'M_Diarrhea'}

		filtered_ihds_individuals_data = filtered_ihds_individuals_data[columns_to_keep_individuals]
		filtered_ihds_individuals_data = filtered_ihds_individuals_data.rename(columns_rename_dict_individuals, axis='columns')

		individuals_data = filtered_ihds_individuals_data.dropna()

		gender_dict = {1:'Male', '1':'Male', 2:'Female', '2':'Female'}
		individuals_data['SexLabel'] = individuals_data['SexLabel'].map(gender_dict)

		# individuals_data.loc[individuals_data['marital_status']==' ','marital_status'] = 1
		# individuals_data['marital_status'] = individuals_data['marital_status'].astype(int)
		# marital_dict = {0:'married', 1:'married', 2:'unmarried', 3:'widowed', 4: 'separated', 5: 'married'}
		# individuals_data['marital_status'] = individuals_data['marital_status'].map(marital_dict)

		# individuals_data.loc[individuals_data['literacy']==' ','literacy'] = 0
		# individuals_data['activity_status'] = individuals_data['activity_status'].astype(int)
		# individuals_data.loc[(individuals_data['literacy']==' ') & (individuals_data['activity_status'] >=5) & (individuals_data['activity_status']<=10),'literacy']=1
		# individuals_data.loc[(individuals_data['literacy']==' ') & (individuals_data['activity_status'] ==12) & (individuals_data['age']!='0to4'),'literacy']=1

		# individuals_data['literacy'] = individuals_data['literacy'].astype(int)
		# individuals_data.loc[individuals_data.literacy == 1, 'literacy'] = 'literate'
		# individuals_data.loc[individuals_data.literacy == 0, 'literacy'] = 'illiterate'

		bins= [0,5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,110]
		labels = ['0to4', '5to9', '10to14', '15to19','20to24', '25to29','30to34', '35to39', '40to44', '45to49',
				 '50to54', '55to59', '60to64', '65to69', '70to74', '75to79', '80p']

		individuals_data['Age'] = individuals_data['Age'].apply(lambda x : try_convert(x, np.float('nan'), int) )   

		individuals_data['Age'] = pd.cut(individuals_data['Age'], bins=bins, labels=labels, right=False)

		religion_dict = {1: 'hindu', 2:'muslim', 3:'christian', 4:'sikh', 5:'buddhist', 6:'jain',
						7: 'other', 8:'other', 9:'other'}
		individuals_data['religion'] = individuals_data['religion'].map(religion_dict)

		individuals_data.loc[individuals_data['caste']==' ', 'caste'] =  random.randint(1,6)
		individuals_data.loc[(individuals_data['caste']==' ') & (individuals_data['religion']!='hindu'),'caste'] = 6
		individuals_data['caste'] = individuals_data['caste'].astype(int)
		caste_dict = {4: 'SC', 5:'ST', 1:'other', 2:'other', 3:'other', 6:'other'}
		individuals_data['caste'] = individuals_data['caste'].map(caste_dict)

		# urbandict = {1:'urban', 0:'rural'}
		# individuals_data['residence'] = individuals_data['residence'].map(urbandict)

		# individuals_data['working'] = 'yes'
		# individuals_data['activity_status'] = individuals_data['activity_status'].apply(lambda x : try_convert(x, np.float('nan'), int) )   
		# individuals_data.loc[individuals_data.activity_status >= 10, 'working'] = 'no'

		individuals_data = individuals_data.drop(['job', 'activity_status', 'edu_years'], axis=1, errors='ignore')

		# individuals_data.loc[individuals_data['literacy']=='illiterate','edu_cat'] = 'illiterate'
		# individuals_data['edu_cat'] = individuals_data['edu_cat'].astype(str)
		# individuals_data.loc[individuals_data['edu_cat']=='0','edu_cat'] = 'illiterate'
		# edu_dict = {'3': 'below_primary', '5':'primary', '8':'middle', '10':'secondary', '12':'senior_secondary',
				#    '15':'grad_p', '16':'grad_p'}
		# individuals_data['edu_cat'].replace(edu_dict, inplace=True)

		return individuals_data

	def preprocess_household_samples_ihds(self, filtered_ihds_households_data):

		columns_to_keep_households = ['DIST01', 'IDHH', 'NPERSONS']
		columns_rename_dict_households = {'URBAN2011':'residence', 'IDHH':'serialno','DIST01':'district', 'NPERSONS':'hhsize'}

		households_data = filtered_ihds_households_data[columns_to_keep_households]
		households_data = households_data.rename(columns_rename_dict_households, axis='columns')

		# urbandict = {1:'urban', 0:'rural'}
		# households_data['residence'] = households_data['residence'].map(urbandict)

		hhsize_dict = {1:'hhsize_1', 2:'hhsize_2', 3:'hhsize_3', 4:'hhsize_4', 5:'hhsize_5',
					  6:'hhsize_6', 7:'hhsize_710', 8:'hhsize_710', 9:'hhsize_710',
					  10:'hhsize_710', 11:'hhsize_1114', 12:'hhsize_1114', 13:'hhsize_1114',
					  14:'hhsize_1114'}

		households_data.loc[households_data['hhsize'] >=15, 'hhsize'] = 'hhsize_15p'
		households_data['hhsize'] = households_data['hhsize'].replace(hhsize_dict)
		return households_data

	def load_marginals(self, householdh_marginal_filename, individuals_marginal_filename, individuals_data, households_data):
		empty_file_households = StringIO("1,2,3") #Creating empty files so that load_data function can be used which is built to load samples as well
		empty_file_individuals = StringIO("1,2,3") #Creating empty files so that load_data function can be used which is built to load samples as well

		household_marginal, individuals_marginal, hh_sample_empty, p_sample_empty, xwalk = load_data(householdh_marginal_filename, individuals_marginal_filename, empty_file_households, empty_file_individuals)

		display(household_marginal)
		display(individuals_marginal)
		household_marginal = household_marginal[list(household_marginal.columns)].astype(float)
		household_marginal = household_marginal[list(household_marginal.columns)].astype(int)

		individuals_marginal = individuals_marginal[list(individuals_marginal.columns)].astype(float)
		individuals_marginal = individuals_marginal[list(individuals_marginal.columns)].astype(int)

		district_dict = pd.Series(individuals_marginal.index, index=individuals_marginal.distid.distid.values).to_dict()
		individuals_data['district'] = individuals_data['district'].replace(district_dict)
		households_data['district'] = households_data['district'].replace(district_dict)
		households_data['sample_geog'] = 1
		individuals_data['sample_geog'] = 1

		household_marginal.drop('distid', axis=1, inplace=True)

		individuals_marginal = individuals_marginal.drop(['distid','total_pop'], axis=1)
		# individuals_marginal = individuals_marginal.drop(['illiterate_males','illiterate_females', 
							#  'literate_males', 'literate_females',
							#  'marginal_less3', 'marginal_6', 'non_worker'], axis=1, level=1)
		# individuals_marginal = individuals_marginal.rename({'main_workers': 'yes', 'non_worker2': 'no'}, axis='columns', level=1)

		# individuals_marginal[('marital_status','separated')] = (individuals_marginal['marital_status']['separated'] + individuals_marginal['marital_status']['divorced']).values

		# individuals_marginal[('edu_cat','senior_secondary')] = (individuals_marginal['edu_cat']['senior_secondary'] + individuals_marginal['edu_cat']['dip_cert_nontech'] + individuals_marginal['edu_cat']['dip_cert_tech']).values
		# individuals_marginal[('edu_cat','illiterate')] = (individuals_marginal['edu_cat']['illiterate'] + individuals_marginal['edu_cat']['lit_wo_edu']).values

		# individuals_marginal.drop(['divorced','dip_cert_nontech', 'dip_cert_tech', 'lit_wo_edu'], axis=1, level=1, inplace=True)

		# individuals_marginal = individuals_marginal.drop(['marital_status', 'edu_cat'], axis=1)

		district_not_in_survey = [] ####### remove rows based on data. This step needs to be adjusted when we add many rows to the marginal file.
		xwalk_dict = dict(xwalk)
		xwalk_dict = {key: xwalk_dict[key] for key in xwalk_dict if key not in district_not_in_survey}
		xwalk = list(tuple(xwalk_dict.items()))

		return household_marginal, individuals_marginal, households_data, individuals_data, xwalk

	def generate_data(self, filtered_ihds_individuals_data, filtered_ihds_households_data, householdh_marginal_filename, individuals_marginal_filename):
		individuals_data = self.preprocess_individual_samples_ihds(filtered_ihds_individuals_data)

		households_data = self.preprocess_household_samples_ihds(filtered_ihds_households_data)

		household_marginal, individuals_marginal, households_data, individuals_data, xwalk = self.load_marginals(householdh_marginal_filename, individuals_marginal_filename, individuals_data, households_data)
	
		individuals_data.dropna(inplace=True)
		households_data.dropna(inplace=True)
  
		# print("\n\n\n Check...\n\n")
		# print(individuals_data, households_data)

		# print(household_marginal, individuals_marginal)
  
		#Dropping columns which lead to error

		
		# household_marginal.drop(['num_workers','hhsize'], axis=1, errors='ignore', inplace=True)
		# households_data.drop(['num_workers','hhsize'], axis=1, errors='ignore', inplace=True)

		# print(household_marginal, households_data, "\n\nCheck\n\n\n")
  

		synthetic_households, synthetic_individuals, synthetic_stats = synthesize_all_zones(household_marginal, individuals_marginal, households_data, individuals_data, xwalk)
		synthetic_households['household_id'] = synthetic_households.index
		return synthetic_households, synthetic_individuals, synthetic_stats

class PopulationDensitySampler:
	def __init__(self, population_density_filename):
		self.population_density_data = pd.read_csv(population_density_filename)
		columns_rename = {"X":"longitude", "Y":"latitude", "Z":"population_density"}
		self.population_density_data['X'] = self.population_density_data['X'].round(6)
		self.population_density_data['Y'] = self.population_density_data['Y'].round(6)
		self.population_density_data.rename(columns_rename, axis=1, inplace=True)
		self.population_density_data['point_object'] = self.population_density_data.progress_apply(lambda x : Point(x['longitude'], x['latitude']), axis=1)

	def add_point(self, latitude, longitude):
		distances = pow(self.population_density_data['latitude']-latitude, 2) + pow(self.population_density_data['longitude']-longitude,2)
		sorted_df = self.population_density_data.loc[distances.sort_values().index]
		mean_population_density = sorted_df.iloc[:4]['population_density'].mean()
		
		new_row_index = len(self.population_density_data)
		
		self.population_density_data.at[new_row_index, 'longitude'] =  longitude
		self.population_density_data.at[new_row_index, 'latitude'] = latitude
		self.population_density_data.at[new_row_index, 'population_density'] = mean_population_density
		self.population_density_data.at[new_row_index, 'point_object'] = Point(longitude, latitude)

	def get_lat_long_samples(self, n, polygon):
		subset = self.population_density_data[self.population_density_data['point_object'].progress_apply(polygon.contains)]
		
		if(len(subset)==0):
			raise Exception("No points within the given polygon")
		
		sample = subset.sample(weights='population_density', n=(n*10), replace=True).copy()
		
		sample.reset_index(drop=True, inplace=True)
		
		sample['latitude'] = sample['latitude'] + np.random.uniform(-0.015, 0.015, size=sample.shape[0])
		
		sample['longitude'] = sample['longitude'] + np.random.uniform(-0.015, 0.015, size=sample.shape[0])
		
		points = sample.progress_apply(lambda x : Point(x['longitude'], x['latitude']), axis=1)
		
		contained = points.progress_apply(polygon.contains)
		
		return sample[contained][['longitude', 'latitude']].sample(n, replace=True).values

class HLatHlongAgeAddition:
	def __init__(self, admin_units_geojson_filename, admin_units_population_filename, population_density_filename):
		self.admin_units = gpd.read_file(admin_units_geojson_filename)
		self.admin_units.sort_values(by='name', inplace=True)
		self.admin_units.reset_index(drop=True, inplace=True)
		self.population_density_sampler = PopulationDensitySampler(population_density_filename)

		self.admin_unit_wise_population = pd.read_csv(admin_units_population_filename)

		self.admin_unit_wise_population['lower_limit'] = self.admin_unit_wise_population['TOT_P'].cumsum()-self.admin_unit_wise_population['TOT_P']
		self.admin_unit_wise_population['upper_limit'] = self.admin_unit_wise_population['TOT_P'].cumsum()

		for admin_unit in self.admin_units.iterrows():
			admin_unit_centroid = admin_unit[1]['geometry'].centroid
			self.population_density_sampler.add_point(admin_unit_centroid.y, admin_unit_centroid.x)

		self.total_population = int(np.ceil(self.admin_unit_wise_population['TOT_P'].sum()/10000)*10000)

	def perform_transforms(self, synthetic_population, synthetic_households):
		synthetic_population['Age'] = synthetic_population['Age'].apply(lambda x : random.randint(80,95) if (x=="80p") else int(x.split("to")[0])) + np.random.randint(0,5,size=len(synthetic_population))

		synthetic_households['hhsize'] = synthetic_population.groupby('household_id').size()

		for admin_unit_wise_population_info in self.admin_unit_wise_population.iterrows():
			subset_index = (synthetic_households['hhsize'].cumsum()>=admin_unit_wise_population_info[1]['lower_limit']) & (synthetic_households['hhsize'].cumsum()<=admin_unit_wise_population_info[1]['upper_limit'])
			synthetic_households.loc[subset_index, 'AdminUnitName'] = admin_unit_wise_population_info[1]['Name']
			synthetic_households.loc[subset_index, 'AdminUnitLatitude'] = admin_unit_wise_population_info[1]['Latitude']
			synthetic_households.loc[subset_index, 'AdminUnitLongitude'] = admin_unit_wise_population_info[1]['Longitude']

		synthetic_households.dropna(inplace=True)

		synthetic_households[['H_Lat', 'H_Lon']] = None

		for admin_unit_name in synthetic_households['AdminUnitName'].unique():
			print(admin_unit_name)
			admin_unit_polygon = self.admin_units[self.admin_units['name']==admin_unit_name]['geometry'].iloc[0]
			admin_unit_houses_index = synthetic_households['AdminUnitName']==admin_unit_name
			n_houses_in_admin_unit = len(synthetic_households[admin_unit_houses_index])
			points = self.population_density_sampler.get_lat_long_samples(n_houses_in_admin_unit, admin_unit_polygon)
			synthetic_households.loc[admin_unit_houses_index, ['H_Lon', 'H_Lat']] = points

		synthetic_households.index.name = 'hh_index'

		columns_to_join = ['household_id', 'H_Lat', 'H_Lon', 'AdminUnitName', 'AdminUnitLatitude', 'AdminUnitLongitude']
		merged_df = pd.merge(synthetic_population, synthetic_households[columns_to_join] ,on='household_id')
		
		return merged_df

class JobsPlacesAddition:
	def __init__(self, job_type_list, admin_units_geojson_filename, n_workplaces, n_public_places, population_density_filename):
		self.job_type_list = job_type_list

		self.admin_units = gpd.read_file(admin_units_geojson_filename)
		self.admin_units.sort_values(by='name', inplace=True)
		self.admin_units.reset_index(drop=True, inplace=True)
		self.combined_boundary = unary_union(self.admin_units['geometry'])

		self.population_density_sampler = PopulationDensitySampler(population_density_filename)

		self.n_workplaces = n_workplaces
		self.n_public_places = n_public_places
		self.city_id = 1
		self.generate_places()
		
	
	def generate_workplaces(self):
		if(self.n_workplaces>len(list(set(self.job_type_list)))):
			random_workplace_types = np.random.choice(list(set(self.job_type_list)), self.n_workplaces-len(list(set(self.job_type_list))), replace=True)
			workplace_types = list(random_workplace_types)+list(set(self.job_type_list))+['Teachers']
		else:
			workplace_types = list(set(self.job_type_list))+['Teachers']
		print(len(workplace_types))
		lat_lon_pairs = self.population_density_sampler.get_lat_long_samples(len(workplace_types), self.combined_boundary)
		workplace_lats = lat_lon_pairs.T[1]
		workplace_longs = lat_lon_pairs.T[0]
		workplace_names = [2*pow(10,12)+self.city_id*pow(10,9)+counter for counter in range(len(workplace_types))]
		self.workplaces = pd.DataFrame([workplace_names, workplace_types, workplace_lats, workplace_longs]).T
		self.workplaces.columns = ['WorkplaceID', 'JobType', 'W_Lat', 'W_Lon']
		self.workplaces.sort_values(by='JobType', inplace=True)
		self.workplaces.reset_index(inplace=True, drop=True)
		
	def generate_schools(self):
		teachers_workplaces = self.workplaces[self.workplaces['JobType']=='Teachers'].copy()
		self.schools = pd.DataFrame([teachers_workplaces['WorkplaceID'], teachers_workplaces['W_Lat'], teachers_workplaces['W_Lon']]).T
		self.schools.columns = ['SchoolID', 'School_Lat', 'School_Lon']
		self.schools.reset_index(inplace=True, drop=True)
		self.schools['SchoolType'] = 'school'

	def generate_public_places(self):
		public_places_number = self.n_public_places
		lat_lon_pairs = self.population_density_sampler.get_lat_long_samples(public_places_number, self.combined_boundary)
		public_place_lats = lat_lon_pairs.T[1]
		public_place_longs = lat_lon_pairs.T[0]
		public_place_names = [3*pow(10,12)+self.city_id*pow(10,9)+counter for counter in range(public_places_number)]
		public_place_types = np.random.choice(['park', 'mall', 'gym'], public_places_number, replace=True)
		self.public_places = pd.DataFrame([public_place_names, public_place_types, public_place_lats, public_place_longs]).T
		self.public_places.columns = ['PublicPlaceID', 'PublicPlaceType', 'PublicPlaceLat', 'PublicPlaceLong']
	
	def generate_places(self):
		self.generate_workplaces()
		self.generate_schools()
		self.generate_public_places()

	def assign_workplaces(self, adult_synthetic_population):
		individuals = adult_synthetic_population
		indicies = []
		workplaceids = []
		for group in tqdm(individuals.groupby(['JobType', 'WorksAtSameCategory'])):
			current_individuals_set = individuals[(individuals['JobType']==group[0][0])&(individuals['WorksAtSameCategory']==group[0][1])]
			if(group[0][1]):
				current_workplaces_set = self.workplaces[self.workplaces['JobType']==group[0][0]]
			else:
				current_workplaces_set = self.workplaces[self.workplaces['JobType']!=group[0][0]]
			individuals_geocode = current_individuals_set[['H_Lat', 'H_Lon']].values.astype(np.float32)
			workplaces_geocode = current_workplaces_set[['W_Lat', 'W_Lon']].values.astype(np.float32)
			place_index = get_probabilistic_place_assignment(individuals_geocode, workplaces_geocode)
			indicies.append(current_individuals_set.index.values)
			workplaceids.append(current_workplaces_set['WorkplaceID'].reset_index(drop=True).iloc[place_index].values)
		individuals['WorkplaceID'] = pd.Series(np.concatenate(workplaceids), index=np.concatenate(indicies))
		individuals = pd.merge(individuals, self.workplaces[['WorkplaceID', 'W_Lat', 'W_Lon']] ,on='WorkplaceID')
		return individuals
	
	def assign_schools(self, child_synthetic_population):
		individuals_geocode = child_synthetic_population[['H_Lat', 'H_Lon']].values.astype(np.float32)
		schools_geocode = self.schools[['School_Lat', 'School_Lon']].values.astype(np.float32)
		school_index = get_probabilistic_place_assignment(individuals_geocode, schools_geocode)
		child_synthetic_population['SchoolID'] = pd.Series(self.schools['SchoolID'].iloc[school_index].values, index=child_synthetic_population.index)
		child_synthetic_population = pd.merge(child_synthetic_population, self.schools[['SchoolID', 'School_Lat', 'School_Lon']], on='SchoolID')
		return child_synthetic_population
	
	def assign_public_places(self, synthetic_population):
		individuals_geocode = synthetic_population[['H_Lat', 'H_Lon']].values.astype(np.float32)
		public_places_geocode = self.public_places[['PublicPlaceLat', 'PublicPlaceLong']].values.astype(np.float32)
		public_places_index = get_probabilistic_place_assignment(individuals_geocode, public_places_geocode)
		synthetic_population['PublicPlaceID'] = pd.Series(self.public_places['PublicPlaceID'].iloc[public_places_index].values, index=synthetic_population.index)
		synthetic_population = pd.merge(synthetic_population, self.public_places[['PublicPlaceID', 'PublicPlaceLat', 'PublicPlaceLong']], on='PublicPlaceID')
		return synthetic_population
	
	def perform_transforms(self, synthetic_population):
		adults = synthetic_population[synthetic_population['Age']>18]
		adults['JobType'] = np.random.choice(self.job_type_list, size=(adults.shape[0],))
		adults['WorksAtSameCategory'] = np.random.uniform(size=(adults.shape[0],))>0.05
		adults = self.assign_workplaces(adults)
  
		gc.collect()
		children = synthetic_population[synthetic_population['Age']<19]
		children['JobType'] = 'Student'
		children['WorksAtSameCategory'] = True
		children = self.assign_schools(children)
		gc.collect()

		total_population = pd.concat([adults,children], axis=0)
		total_population = self.assign_public_places(total_population)
		return total_population

In [ ]:
districts = ['ambala',
 'bhiwani',
 'fatehabad',
 'gurgaon',
 'hisar',
 'jhajjar',
 'jind',
 'kaithal',
 'karnal',
 'kurukshetra',
 'mahendragarh',
 'panchkula',
 'panipat',
 'rewari',
 'rohtak',
 'sirsa',
 'yamunanagar',
 'sonipat']

for district in districts:
    ###Configuration

    region_name = "haryana/"+district
    state_id = 6 ####### select state number and name appropriately
    district_ids = [] #Mumbai Suburban and Mumbai Urban

    district_wise_filtering = False

    population_density_filename = "ind_pd_2020_1km_ASCII_XYZ.csv"

    working_dir = "/content/drive/MyDrive/data/synthetic/"

    ihds_individuals_filename = working_dir+"36151-0001-Data.tsv"
    ihds_household_filename = working_dir+"36151-0002-Data.tsv"

    householdh_marginal_filename = working_dir+region_name+"/household_marg.csv"
    individuals_marginal_filename = working_dir+region_name+"/person_marg.csv"

    admin_units_geojson_filename = working_dir+region_name+"/admin_units.geojson"
    admin_units_population_filename = working_dir+region_name+"/admin_unit_wise_pop.csv"

    if(district_wise_filtering):
        synthetic_population_output_filename = working_dir+region_name+"/synthetic.csv"
    else:
        synthetic_population_output_filename = working_dir+region_name+"/synthetic_state_wise.csv"
    
    if(not os.path.exists(synthetic_population_output_filename)):

        ihds_individuals_data = pd.read_csv(ihds_individuals_filename, sep='\t')

        filtered_ihds_individuals_data = ihds_individuals_data.loc[ihds_individuals_data.STATEID==state_id]

        ihds_households_data = pd.read_csv(ihds_household_filename, sep='\t')

        filtered_ihds_households_data = ihds_households_data.loc[ihds_households_data.STATEID==state_id] 

        if(district_wise_filtering):
            filtered_ihds_individuals_data = filtered_ihds_individuals_data[filtered_ihds_individuals_data['DISTID'].apply(lambda x : x in district_ids)]   
            filtered_ihds_households_data = filtered_ihds_households_data[filtered_ihds_households_data['DISTID'].apply(lambda x : x in district_ids)]

        ipu_object = IPU()

        synthetic_households, synthetic_individuals, synthetic_stats = ipu_object.generate_data(filtered_ihds_individuals_data, filtered_ihds_households_data, householdh_marginal_filename, individuals_marginal_filename)


        hlat_hlong_age_object = HLatHlongAgeAddition(admin_units_geojson_filename, admin_units_population_filename, population_density_filename)

        new_synthetic_population = hlat_hlong_age_object.perform_transforms(synthetic_individuals, synthetic_households)


        n_workplaces = (len(new_synthetic_population)//100) + np.random.randint(1,100)
        n_publicplaces = (len(new_synthetic_population)//200) + np.random.randint(1,100)

        jobs_places_object = JobsPlacesAddition(POSSIBLE_JOB_LABELS, admin_units_geojson_filename, n_workplaces, n_publicplaces, population_density_filename)

        new_synthetic_population = jobs_places_object.perform_transforms(new_synthetic_population) 

        new_synthetic_population.to_csv(synthetic_population_output_filename, index=False)

        del(new_synthetic_population)
        del(synthetic_households)
        del(synthetic_individuals)

/usr/local/lib/python3.7/dist-packages/IPython/core/interactiveshell.py:2882: DtypeWarning: Columns (0,6,9,10,11,12,13,14,15,16,17,18,22,27,29,33,34,39,44,48,49,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,74,75,76,78,79,80,88,91,105,150,151,152,154,158,159,166,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,194,196,197,198,199,200,201,202,203,204,205,217,218,219,220,221,222,223,224,225,226,227,228,229,230,234,235,236,237,240,241,243,245,247,248,249,250,275,279,283,287,288,289,290,291,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,320,322,334) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
/usr/local/lib/python3.7/dist-packages/IPython/core/interactiveshell.py:2882: DtypeWarning: Columns (14,15,16,17,18,19,20,21,24,25,26,28,31,35,36,113,114,115,116,117,118,119,120,121,122,126,186,187,188,189,190,191,192,193,194,212,213,214

district             distid   hhsize                                      \
         Unnamed: 1_level_1 hhsize_1 hhsize_2 hhsize_3 hhsize_4 hhsize_5   
sonipat                  76   7443.0  15092.0  27014.0  63468.0  62463.0   

district                                             
         hhsize_6 hhsize_710 hhsize_1114 hhsize_15p  
sonipat   42727.0    49950.0      6719.0     1533.0

district distid  total_pop  SexLabel                 Age                      \
         distid  total_pop      Male    Female      0to4      5to9    10to14   
sonipat      76  1450001.0  781299.0  668702.0  131531.0  139770.0  149734.0   

district                                ...  religion                    \
            15to19    20to24    25to29  ...     hindu  muslim christian   
sonipat   155325.0  150360.0  128481.0  ...  170391.0  6867.0     281.0   

district                                  caste              
             sikh buddhist  jain  other      SC ST    other  
sonipat   19746.0     37.0  16.0  447.0  269935  0  1180066  

[1 rows x 31 columns]

/usr/local/lib/python3.7/dist-packages/pandas/core/generic.py:4150: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  obj = obj._drop_axis(labels, axis, level=level, errors=errors)
/content/synthpop/synthpop/ipu/ipu.py:193: RuntimeWarning: divide by zero encountered in double_scalars
  adj = constraint / float((column * weights).sum())


Drawing 276409 households


100%|█████████▉| 4009983/4010402 [02:01<00:00, 35248.00it/s]/usr/local/lib/python3.7/dist-packages/pandas/core/dtypes/cast.py:118: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  arr = construct_1d_object_array_from_listlike(values)
100%|██████████| 4010402/4010402 [03:38<00:00, 18384.27it/s]


sonipat


100%|█████████▉| 2762575/2764090 [01:21<00:00, 34108.55it/s]/usr/local/lib/python3.7/dist-packages/pandas/core/dtypes/cast.py:118: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  arr = construct_1d_object_array_from_listlike(values)
100%|█████████▉| 4008370/4010402 [01:59<00:00, 35688.24it/s]/usr/local/lib/python3.7/dist-packages/pandas/core/dtypes/cast.py:118: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  arr = construct_1d_object_array_from_listlike(values)
100%|██████████| 4010402/4010402 [03:38<00:00, 18321.55it/s]


14420


100%|██████████| 72210/72210 [00:02<00:00, 30953.60it/s]
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:393: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:394: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
100%|██████████| 186/186 [00:33<00:00,  5.48it/s]
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:371: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,